## Context engineering in Deep Agents
Context engineering is providing the right information and tools in the right format so your deep agent can accomplish tasks reliably.
Deep agents have access to several kinds of context. Some sources are provided to the agent at startup; others become available during runtime, such as user input. Deep agents include built-in mechanisms for managing context across long-running sessions.



In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["Deepseek_API_KEY"]=os.getenv("Deepseek_API_KEY")
print("Keys loaded!")


Keys loaded!


In [23]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from typing import Literal
from langchain_deepseek import ChatDeepSeek
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.agents.middleware import SummarizationMiddleware

# 1. Load API Keys
load_dotenv()

# 2. Set up the Web Search Tool
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def web_search(query: str, max_results: int = 5, topic: Literal["general", "sports", "news", "finance"] = "general", include_raw_content: bool = False):
    """Run a web search"""
    return tavily_client.search(
        query=query,
        max_results=max_results,
        topic=topic,
        include_raw_content=include_raw_content
    )

# 3. Initialize the Model
model = ChatDeepSeek(model="deepseek-v4-pro")

# 4. Set up the Real-Disk Backend
backend = FilesystemBackend(root_dir=".", virtual_mode=True)

# 5. Set up Summarization Middleware (To prevent token drain)
summarization_middleware = SummarizationMiddleware(
    model=model,
    trigger=("fraction", 0.8), 
    keep=("tokens", 3000)
)

# 6. Define the Elite Research System Prompt
elite_researcher_prompt = """
You are the Chief Scientific Officer and Principal Systems Architect at an elite, multidisciplinary research institute. Your expertise spans the absolute cutting edge of: Artificial Intelligence (AI), Machine Learning (ML), Data Science (DS), Artificial General Intelligence (AGI), Unmanned Air Systems (UAS/Drones), Wireless Communications, and Strategic Touring/Reconnaissance.

Your mandate is to conduct high-level, rigorous, and actionable research. You do not give superficial answers. You think like a PhD-level researcher, an engineer, and a strategist simultaneously.

## CORE METHODOLOGY (The Scientific Method Loop)
When given a research task, you MUST execute the following loop:
1. DECONSTRUCT & PLAN: Break the complex problem into sub-tasks. Write this plan down.
2. DEEP SEARCH: Use your `web_search` tool extensively. Search for recent whitepapers, arXiv papers, IEEE standards, defense briefings, and SOTA (State of the Art) benchmarks.
3. CRITICAL ANALYSIS: Do not trust a single source. Cross-reference data. Identify gaps in the current literature. Look for edge cases and systemic risks.
4. SIMULATE & CODE: Where applicable, write Python scripts to model the problem, simulate drone swarms, analyze datasets, or test ML architectures.
5. DOCUMENT & PERSIST: Use your filesystem tools to save your work. 
   - Save code to `/code/`
   - Save datasets or raw data summaries to `/data/`
   - Save final reports, whitepapers, and strategic briefs to `/research/`

## DOMAIN-SPECIFIC DIRECTIVES
- **AI / ML / DS / AGI:** Focus on architectures (Transformers, SSMs, RL), scaling laws, alignment, emergent behaviors, and hardware constraints. Critique current models.
- **UAS / Drones / Wireless Systems:** Focus on RF propagation, 5G/6G integration, edge computing, swarm intelligence, latency, control theory (MPC, PID), and electronic warfare (jamming/spoofing).
- **Strategic Touring/Reconnaissance:** Focus on optimal path planning, dynamic environmental constraints, energy efficiency, and logistics.

## OUTPUT FORMATTING RULES
- Always use Markdown for formatting.
- When presenting code, use proper syntax highlighting (e.g., ```python).
- Structure your final responses with: Executive Summary, Technical Deep Dive, Data/Simulation Results, and Strategic Recommendations.
- Always cite your sources using URLs when providing factual research.
"""

# 7. Create the Agent
agent = create_deep_agent(
    model="deepseek-v4-pro",
    system_prompt=elite_researcher_prompt,
    tools=[web_search],
    backend=backend,
    middleware=[summarization_middleware]
)

print("✅ Elite Research Agent is ready!")

✅ Elite Research Agent is ready!


In [9]:
# Invoke the agent with a research task
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Research the latest advancements in drone swarm AI for 2026. Save a summary report to /research/drone_swarms_2026.md and write a basic Python simulation script to /code/swarm_sim.py."
    }]
})

# Print the final response
print(result["messages"][-1].content)

Both deliverables are saved and verified. Here's a summary of the completed work.

## Deliverables

| Artifact | Path | Status |
|---|---|---|
| Research report (Markdown) | `/research/drone_swarms_2026.md` | ✅ Saved |
| Python swarm simulation | `/code/swarm_sim.py` | ✅ Saved & logic-verified |

## Research Highlights (2026 State of the Art)

**The single biggest shift:** Multi-agent reinforcement learning (MARL) — **MADDPG, MAPPO, QMIX** — has displaced bio-inspired heuristics (PSO, ACO) as the dominant coordination paradigm, per the PatSnap 2026 patent landscape ([source](https://www.patsnap.com/resources/blog/articles/drone-swarm-coordination-patent-landscape-2026)).

Five structural trends I synthesized across ~25 cross-referenced sources:

1. **MARL as coordination default** — Korea leads patent filings (ETRI's MADDPG-per-agent Markov games; Hanwha's QMIX-style mixing networks with priority replay). CTDE (centralized training, decentralized execution) is the practical sweet spot 

In [11]:
from pathlib import Path

# Read the AGENTS.md file directly from the current folder
agents_md = Path("AGENTS.md").read_text(encoding="utf-8")
agents_md

'# Agents.md — Deep Agents Context File\n\n> This file is loaded into the deep agent\'s context (via its file system tools)\n> whenever the agent is invoked. It gives the agent background knowledge about\n> its own architecture, the project conventions, and how it should operate.\n\n---\n\n## 1. What is a Deep Agent?\n\nA **deep agent** is an agent designed for complex, multi-step, long-horizon\ntasks. Unlike a simple tool-calling loop (LLM → tool → LLM → answer), a deep\nagent is built on **LangGraph** and ships with four architectural pillars,\ninspired by applications like Claude Code, Deep Research, and Manus:\n\n1. **Planning** — an explicit todo-list tool the agent uses to break a task\n   into steps and track progress.\n2. **File System (Context Offloading)** — virtual file tools so the agent can\n   store large intermediate results outside the chat context window.\n3. **Subagents** — the ability to spawn specialized child agents with their\n   own isolated context, tools, and p

Context engineering in Deep Agents


Context engineering is providing the right information and tools in the right format so your deep agent can accomplish tasks reliably.Deep agents have access to several kinds of context. Some sources are provided to the agent at startup; others become available during runtime, such as user input. Deep agents include built-in mechanisms for managing context across long-running sessions.

In [13]:
from pathlib import Path
from deepagents import create_deep_agent

# 1. Read the AGENTS.md file (Context Engineering!)
agents_md = Path("AGENTS.md").read_text(encoding="utf-8")

# 2. Create the agent using the content of AGENTS.md as the system prompt
agent = create_deep_agent(
    model=model,                          # Your initialized DeepSeek model
    system_prompt=agents_md,              # <--- The file content is used here
    tools=[web_search],                   # Your Tavily web search tool
    backend=backend,                      # Your FilesystemBackend
    middleware=[summarization_middleware] # Your token-saving middleware
)

print("✅ Agent successfully created using AGENTS.md!")

✅ Agent successfully created using AGENTS.md!


In [20]:
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model="deepseek-v4-pro", # i am using deepseek
    checkpointer=MemorySaver(),
)

# 2. Inject the AGENTS.md content directly into the message
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": f"Here is our project guide:\n\n{agents_md}\n\nNow, who are you and what should you follow?"}
        ]
    },
    config={"configurable": {"thread_id": "default-demo-1"}}, 
)

print(result["messages"][-1].content)

I am the **Deep Agent** described in `Agents.md` — an agent built on **LangGraph** with four pillars:

1. **Planning** — a `write_todos` tool to break tasks into steps and track progress.
2. **File System** — `ls`, `read_file`, `write_file`, `edit_file` for context offloading to a virtual file system.
3. **Subagents** — a `task` tool to delegate work to isolated child agents.
4. **Detailed System Prompt** — instructions on when and how to use those capabilities.

What I should follow:

- **Plan first** — for any task with more than ~2 steps, write and maintain a todo list before doing the work.
- **Offload aggressively** — write bulky intermediate results (research notes, scraped pages, long drafts) to files, keeping the conversation lean.
- **Delegate wisely** — use subagents for deep-dive or independent subtasks, with crisp self-contained instructions.
- **Treat `Agents.md` as the one source of truth** about my own architecture and conventions — prefer it over assumptions.
- **Final 